# Extracción de keypoints con YOLO11-Pose

**Proyecto:** detección y clasificación de golpes de boxeo mediante **YOLO11-Pose + LSTM**.

## Objetivo

Procesar los videos de cada clase de golpe para detectar los puntos clave del cuerpo (*keypoints*) en cada fotograma. Los resultados se guardan para utilizarlos posteriormente en el modelo LSTM y para revisar visualmente la calidad de las detecciones.

## Archivos generados

- `salida/keypoints/<clase>/<video>.npz`: coordenadas, confianzas y detección por fotograma.
- `salida/overlays/<clase>/<video>.mp4`: video con el esqueleto dibujado.
- `salida/manifest.csv`: resumen y estado de procesamiento de cada video.
- `salida/balance_clases.png`: cantidad de videos y fotogramas por clase.
- `salida/muestras_esqueleto.png`: ejemplos de las detecciones obtenidas.

**Orden de ejecución:** ejecutar las celdas de arriba hacia abajo. Este notebook está preparado para Google Colab y utiliza una carpeta del proyecto ubicada en Google Drive.

## 1. Instalar e importar las bibliotecas

La siguiente celda instala Ultralytics, que permite utilizar YOLO11-Pose.

In [ ]:
!pip -q install ultralytics

In [ ]:
# Bibliotecas para archivos, videos, datos, gráficos y modelo
import shutil
import subprocess
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from ultralytics import YOLO

# Conectar Google Drive (entorno Google Colab)
from google.colab import drive

drive.mount('/content/drive')

# Carpeta principal del proyecto en Google Drive
BASE = Path('/content/drive/MyDrive/boxeo_proyecto')

print('GPU disponible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print(
        'AVISO: no hay GPU. En Colab, ve a '
        'Entorno de ejecución > Cambiar tipo de entorno > T4 GPU.'
    )

## 2. Configurar el procesamiento

En esta sección se definen el modelo, los formatos de video aceptados, los umbrales para marcar videos que necesitan revisión y las carpetas de entrada y salida.

In [ ]:
# Modelo de detección de pose
MODELO = 'yolo11m-pose.pt'

# Extensiones de video que se procesarán
EXTENSIONES = {'.mp4', '.mov', '.m4v', '.avi'}

# Si es False, se omiten los videos que ya tienen keypoints guardados
REPROCESAR = True

# Criterios para marcar un video como REVISAR
UMBRAL_SIN_DETECCION = 0.20  # Proporción máxima de frames sin persona detectada
UMBRAL_CONF = 0.50           # Confianza media mínima para los puntos seleccionados

# Tamaño máximo del lado más largo de los videos de revisión
LADO_MAX_OVERLAY = 1280

# Índices COCO: hombros, codos, muñecas y caderas
PUNTOS_TORSO = [5, 6, 7, 8, 9, 10, 11, 12]

# Carpetas de entrada y salida
VIDEOS = BASE / 'videos'
SALIDA = BASE / 'salida'
KP_DIR = SALIDA / 'keypoints'
OV_DIR = SALIDA / 'overlays'

# Comprobar que exista la carpeta de videos
if not VIDEOS.exists():
    raise FileNotFoundError(
        f'No encuentro la carpeta {VIDEOS}. '
        'Revisa que exista en Drive con ese nombre exacto.'
    )

# Crear las carpetas de salida si todavía no existen
KP_DIR.mkdir(parents=True, exist_ok=True)
OV_DIR.mkdir(parents=True, exist_ok=True)

# Cada subcarpeta dentro de "videos" representa una clase de golpe
clases = sorted(carpeta.name for carpeta in VIDEOS.iterdir() if carpeta.is_dir())
print('Clases encontradas:', clases)

## 3. Cargar el modelo YOLO11-Pose

In [ ]:
model = YOLO(MODELO)
print('Modelo cargado:', MODELO)

## 4. Definir las funciones de procesamiento

Se utilizan tres funciones:

1. `a_h264`: intenta convertir el video de revisión al formato H.264.
2. `procesar_video`: analiza el video fotograma a fotograma y extrae los keypoints.
3. `estadisticas`: calcula métricas y determina si el video queda como `OK` o `REVISAR`.

In [ ]:

def a_h264(origen: Path, destino: Path):
    """Convierte un video a H.264 para mejorar su compatibilidad."""
    try:
        subprocess.run(
            [
                'ffmpeg', '-y', '-loglevel', 'error',
                '-i', str(origen),
                '-vcodec', 'libx264',
                '-pix_fmt', 'yuv420p',
                str(destino),
            ],
            check=True,
        )
        origen.unlink()

    except Exception as error:
        # Si la conversión falla, conservar el video original.
        shutil.move(str(origen), str(destino))
        print('  (aviso) No se pudo convertir a H.264:', error)


def procesar_video(ruta_video: Path, ruta_overlay: Path):
    """Extrae 8 keypoints y genera un video con esos puntos dibujados."""

    cap = cv2.VideoCapture(str(ruta_video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    ruta_temporal = ruta_overlay.with_name(
        ruta_overlay.stem + '_tmp.mp4'
    )

    writer = None

    keypoints = []
    confianzas = []
    detecciones = []

    muestra = None
    ultimo_frame = None

    while True:
        correcto, frame = cap.read()

        if not correcto:
            break

        resultado = model(frame, verbose=False)[0]

        hay_persona = (
            resultado.keypoints is not None
            and resultado.boxes is not None
            and len(resultado.boxes) > 0
        )

        # Partir del fotograma original, sin el esqueleto completo de YOLO.
        frame_anotado = frame.copy()

        if hay_persona:
            # Si aparecen varias personas, seleccionar la de mayor área.
            cajas = resultado.boxes.xywh
            areas = cajas[:, 2] * cajas[:, 3]
            indice = int(areas.argmax())

            # Obtener los 17 puntos originales detectados por YOLO.
            puntos_originales = (
                resultado.keypoints.xy[indice].cpu().numpy()
            )

            # Conservar únicamente los 8 puntos seleccionados.
            puntos = puntos_originales[PUNTOS_TORSO]
            keypoints.append(puntos.astype(np.float32))

            # Obtener las confianzas de los mismos puntos.
            if resultado.keypoints.conf is not None:
                confianza_original = (
                    resultado.keypoints.conf[indice].cpu().numpy()
                )
                confianza = confianza_original[PUNTOS_TORSO]
                confianzas.append(confianza.astype(np.float32))
            else:
                # Si YOLO no entrega confianzas, usar 1.0.
                confianza = np.ones(
                    len(PUNTOS_TORSO),
                    dtype=np.float32
                )
                confianzas.append(confianza)

            detecciones.append(True)

            # Dibujar solo puntos con confianza >= 0.5.
            for i, (x, y) in enumerate(puntos):
                if (
                    x > 0
                    and y > 0
                    and confianza[i] >= 0.5
                ):
                    cv2.circle(
                        frame_anotado,
                        (int(x), int(y)),
                        5,
                        (0, 255, 0),
                        -1
                    )

        else:
            # Mantener dimensiones consistentes si no hay detección.
            keypoints.append(
                np.zeros(
                    (len(PUNTOS_TORSO), 2),
                    dtype=np.float32
                )
            )

            confianzas.append(
                np.zeros(
                    len(PUNTOS_TORSO),
                    dtype=np.float32
                )
            )

            detecciones.append(False)

        # Reducir el tamaño del video de revisión para ahorrar espacio.
        lado_mayor = max(frame_anotado.shape[:2])

        if lado_mayor > LADO_MAX_OVERLAY:
            escala = LADO_MAX_OVERLAY / lado_mayor

            nuevo_tamano = (
                int(frame_anotado.shape[1] * escala),
                int(frame_anotado.shape[0] * escala),
            )

            frame_anotado = cv2.resize(
                frame_anotado,
                nuevo_tamano,
                interpolation=cv2.INTER_AREA,
            )

        # Inicializar el escritor con el tamaño del primer fotograma.
        if writer is None:
            alto, ancho = frame_anotado.shape[:2]
            codec = cv2.VideoWriter_fourcc(*'mp4v')

            writer = cv2.VideoWriter(
                str(ruta_temporal),
                codec,
                fps,
                (ancho, alto)
            )

        writer.write(frame_anotado)
        ultimo_frame = frame_anotado

        # Guardar el fotograma 30 como muestra de evidencia.
        if len(keypoints) == 30:
            muestra = frame_anotado.copy()

    cap.release()

    if writer is not None:
        writer.release()
        a_h264(ruta_temporal, ruta_overlay)

        # Si el video tiene menos de 30 fotogramas, usar el último.
        if muestra is None:
            muestra = ultimo_frame

    # Convertir los resultados a arreglos NumPy.
    keypoints = (
        np.array(keypoints, dtype=np.float32)
        if keypoints
        else np.zeros(
            (0, len(PUNTOS_TORSO), 2),
            dtype=np.float32
        )
    )

    confianzas = (
        np.array(confianzas, dtype=np.float32)
        if confianzas
        else np.zeros(
            (0, len(PUNTOS_TORSO)),
            dtype=np.float32
        )
    )

    detecciones = np.array(detecciones, dtype=bool)

    return keypoints, confianzas, detecciones, float(fps), muestra


def estadisticas(clase, nombre, keypoints, confianzas, detectado, fps):
    """Calcula métricas del video y asigna el estado OK o REVISAR."""

    cantidad_frames = len(keypoints)

    porcentaje_sin_deteccion = (
        float(1 - detectado.mean())
        if cantidad_frames
        else 1.0
    )

    if detectado.any():
        confianza_media = float(
            confianzas[detectado].mean()
        )
    else:
        confianza_media = 0.0

    requiere_revision = (
        cantidad_frames == 0
        or porcentaje_sin_deteccion > UMBRAL_SIN_DETECCION
        or confianza_media < UMBRAL_CONF
    )

    estado = 'REVISAR' if requiere_revision else 'OK'

    return {
        'clase': clase,
        'video': nombre,
        'frames': cantidad_frames,
        'fps': round(fps, 1),
        'duracion_s': round(cantidad_frames / fps, 2) if fps else 0,
        'pct_sin_deteccion': round(
            porcentaje_sin_deteccion * 100, 1
        ),
        'conf_media_torso': round(confianza_media, 3),
        'estado': estado,
    }


print('Funciones listas.')


## 5. Encontrar y procesar los videos

El notebook recorre las subcarpetas de `videos`, procesa los archivos compatibles y guarda los resultados. Si `REPROCESAR = False`, reutiliza los archivos `.npz` existentes.

In [ ]:
# Crear una lista de pares (clase, ruta del video).
videos = []
for clase in clases:
    carpeta_clase = VIDEOS / clase

    for ruta in sorted(carpeta_clase.iterdir()):
        es_video = ruta.suffix.lower() in EXTENSIONES
        es_archivo_oculto = ruta.name.startswith('.')

        if es_video and not es_archivo_oculto:
            videos.append((clase, ruta))

if not videos:
    raise RuntimeError(
        f'No encontré videos en {VIDEOS}. Revisa que estén dentro de '
        'las subcarpetas de cada clase y que tengan una extensión compatible.'
    )

print(f'Videos encontrados: {len(videos)}')

filas = []
muestras = {}

for clase, ruta_video in tqdm(videos, desc='Procesando videos'):
    nombre = ruta_video.stem

    # Crear subcarpetas para la clase actual.
    (KP_DIR / clase).mkdir(parents=True, exist_ok=True)
    (OV_DIR / clase).mkdir(parents=True, exist_ok=True)

    ruta_keypoints = KP_DIR / clase / f'{nombre}.npz'
    ruta_overlay = OV_DIR / clase / f'{nombre}.mp4'

    # Reutilizar los keypoints guardados si no se solicita reprocesar.
    if ruta_keypoints.exists() and not REPROCESAR:
        datos = np.load(ruta_keypoints)
        keypoints = datos['kpts']
        confianzas = datos['conf']
        detectado = datos['detectado']
        fps = float(datos['fps'])
    else:
        keypoints, confianzas, detectado, fps, muestra = procesar_video(
            ruta_video,
            ruta_overlay,
        )

        np.savez_compressed(
            ruta_keypoints,
            kpts=keypoints,
            conf=confianzas,
            detectado=detectado,
            fps=fps,
        )

        # Guardar una muestra por clase para el gráfico final.
        if muestra is not None and clase not in muestras:
            muestras[clase] = (nombre, muestra)

    filas.append(
        estadisticas(
            clase,
            nombre,
            keypoints,
            confianzas,
            detectado,
            fps,
        )
    )

# Crear el resumen general y exportarlo a CSV.
df = pd.DataFrame(filas)
ruta_manifest = SALIDA / 'manifest.csv'
df.to_csv(ruta_manifest, index=False, encoding='utf-8')

print('Procesamiento terminado. Resumen guardado en:', ruta_manifest)
display(df)

## 6. Generar gráficos de evidencia

Se generan gráficos para comparar la cantidad de videos y fotogramas por clase, además de una imagen de ejemplo con el esqueleto detectado.

In [ ]:
# Resumir la cantidad de videos y frames de cada clase.
resumen = (
    df.groupby('clase')
    .agg(videos=('video', 'count'), frames=('frames', 'sum'))
    .reset_index()
)

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))

ejes[0].bar(resumen['clase'], resumen['videos'])
ejes[0].set_title('Videos por clase')
ejes[0].set_xlabel('Clase')
ejes[0].set_ylabel('Cantidad de videos')

ejes[1].bar(resumen['clase'], resumen['frames'])
ejes[1].set_title('Frames por clase')
ejes[1].set_xlabel('Clase')
ejes[1].set_ylabel('Cantidad de frames')

plt.tight_layout()
plt.savefig(SALIDA / 'balance_clases.png', dpi=150)
plt.show()

# Mostrar una imagen de ejemplo por cada clase procesada en esta ejecución.
if muestras:
    fig, ejes = plt.subplots(
        1,
        len(muestras),
        figsize=(4 * len(muestras), 4),
        squeeze=False,
    )

    for eje, (clase, (nombre, imagen)) in zip(ejes[0], muestras.items()):
        imagen_rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        eje.imshow(imagen_rgb)
        eje.set_title(f'{clase}\n{nombre}', fontsize=9)
        eje.axis('off')

    plt.tight_layout()
    plt.savefig(SALIDA / 'muestras_esqueleto.png', dpi=150)
    plt.show()
else:
    print(
        'No hay muestras nuevas porque los videos ya estaban procesados. '
        'Cambia REPROCESAR = True en la sección 2 si quieres generarlas de nuevo.'
    )

print('Revisa los resultados en:', SALIDA)

## 7. Cómo interpretar los resultados

- **`OK`**: el video cumple los umbrales definidos para detección y confianza.
- **`REVISAR`**: hay demasiados fotogramas sin detección, la confianza media es baja o el video no contiene fotogramas procesables.
- **`kpts`**: coordenadas `(x, y)` de los 8 keypoints seleccionados para cada fotograma.
- **`conf`**: confianza estimada para cada uno de los 8 keypoints seleccionados.
- **`detectado`**: indica si se encontró una persona en cada fotograma.

Los umbrales son configurables y el estado `OK` no garantiza por sí solo que cada keypoint sea perfecto; conviene revisar los videos de salida y los casos marcados como `REVISAR`.